In [ ]:
import sys
from pathlib import Path
import numpy as np
import time

ROOT = Path.cwd()
SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

from src.hand_tracking_module import handDetector

In [2]:
import cv2

cap = cv2.VideoCapture(0)
success, frame = cap.read()
print("Camera working:", success)
cap.release()

Camera working: True


In [3]:
# def main():
#     cap = cv2.VideoCapture(0)
#     detector = handDetector(num_hands=1)

#     xp, yp = 0, 0
#     brush_thickness = 12
#     draw_color = (255, 0, 255)
#     img_canvas = None

#     while True:
#         success, img = cap.read()
#         if not success:
#             print("Camera not found")
#             break

#         img = cv2.flip(img, 1)
#         height, width = img.shape[:2]

#         if img_canvas is None:
#             img_canvas = np.zeros((height, width, 3), np.uint8)

#         img = detector.findHands(img)
#         lm_list = detector.findPosition(img, draw=False)

#         if lm_list:
#             x1, y1 = lm_list[8][1:]
#             fingers = detector.fingersUp()

#             # Index finger only: draw
#             if fingers == [0, 1, 0, 0, 0]:
#                 cv2.circle(img, (x1, y1), 10, draw_color, cv2.FILLED)

#                 if xp == 0 and yp == 0:
#                     xp, yp = x1, y1

#                 cv2.line(img, (xp, yp), (x1, y1), draw_color, brush_thickness)
#                 cv2.line(img_canvas, (xp, yp), (x1, y1), draw_color, brush_thickness)
#                 xp, yp = x1, y1
#             else:
#                 xp, yp = 0, 0

#             # Full open hand: erase
#             if fingers == [1, 1, 1, 1, 1]:
#                 erase_x, erase_y = lm_list[9][1], lm_list[9][2]
#                 cv2.circle(img_canvas, (erase_x, erase_y), 100, (0, 0, 0), cv2.FILLED)

#         img_gray = cv2.cvtColor(img_canvas, cv2.COLOR_BGR2GRAY)
#         _, img_inv = cv2.threshold(img_gray, 50, 255, cv2.THRESH_BINARY_INV)
#         img_inv = cv2.cvtColor(img_inv, cv2.COLOR_GRAY2BGR)

#         img = cv2.bitwise_and(img, img_inv)
#         img = cv2.bitwise_or(img, img_canvas)

#         cv2.putText(
#             img,
#             "Index: draw | Open hand: erase | C: clear | Q: quit",
#             (20, 40),
#             cv2.FONT_HERSHEY_SIMPLEX,
#             0.8,
#             (0, 0, 255),
#             2,
#         )

#         cv2.imshow("Adult Air Writing Test", img)
#         cv2.imshow("Canvas Only", img_canvas)

#         key = cv2.waitKey(1) & 0xFF

#         if key == ord("c"):
#             img_canvas[:] = 0
#             xp, yp = 0, 0

#         if key == ord("q"):
#             break

#     cap.release()
#     cv2.destroyAllWindows()


# if __name__ == "__main__":
#     main()

In [4]:
import torch
from predict import get_class_names, load_model

MNIST_MODEL_PATH = ROOT / "models" / "mnist_cnn.pth"
EMNIST_MODEL_PATH = ROOT / "models" / "emnist_letters_cnn.pth"

digit_model = load_model(str(MNIST_MODEL_PATH), dataset_name="mnist")
digit_classes = get_class_names("mnist")

alphabet_model = load_model(str(EMNIST_MODEL_PATH), dataset_name="emnist_letters")
alphabet_classes = get_class_names("emnist_letters")

# prediction

In [5]:
def prepare_canvas_for_model(img_canvas):
    gray = cv2.cvtColor(img_canvas, cv2.COLOR_BGR2GRAY)

    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)

    ys, xs = np.where(thresh > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None

    crop = thresh[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

    height, width = crop.shape[:2]
    side = max(height, width) + 16

    square = np.zeros((side, side), dtype=np.uint8)
    y_offset = (side - height) // 2
    x_offset = (side - width) // 2
    square[y_offset:y_offset + height, x_offset:x_offset + width] = crop

    resized = cv2.resize(square, (28, 28), interpolation=cv2.INTER_AREA)

    tensor = torch.tensor(resized, dtype=torch.float32) / 255.0
    tensor = tensor.unsqueeze(0).unsqueeze(0)

    return tensor

def predict_word_canvas(img_canvas, model, class_names):
    gray = cv2.cvtColor(img_canvas, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(
        thresh,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE,
    )

    boxes = []

    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)

        # Ignore tiny noise
        if w < 8 or h < 8:
            continue

        boxes.append((x, y, w, h))

    boxes = sorted(boxes, key=lambda box: box[0])

    if not boxes:
        return "", 0.0, []

    word = ""
    confidences = []
    predictions = []

    pad = 20

    for x, y, w, h in boxes:
        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(img_canvas.shape[1], x + w + pad)
        y2 = min(img_canvas.shape[0], y + h + pad)

        char_canvas = np.zeros_like(img_canvas)
        char_canvas[y1:y2, x1:x2] = img_canvas[y1:y2, x1:x2]

        label, confidence = predict_canvas(
            char_canvas,
            model,
            class_names,
        )

        word += label
        confidences.append(confidence)
        predictions.append((label, confidence, (x, y, w, h)))

    avg_confidence = sum(confidences) / len(confidences)

    return word, avg_confidence, predictions

def canvas_has_content(img_canvas):
    if img_canvas is None:
        return False

    gray = cv2.cvtColor(img_canvas, cv2.COLOR_BGR2GRAY)
    return np.count_nonzero(gray) > 50


def predict_canvas(img_canvas, model, class_names):
    tensor = prepare_canvas_for_model(img_canvas)

    if tensor is None:
        return "", 0.0

    with torch.no_grad():
        output = model(tensor)
        probabilities = torch.softmax(output, dim=1)
        confidence, predicted_class = torch.max(probabilities, dim=1)

    label = class_names[predicted_class.item()]
    confidence = confidence.item()

    return label, confidence


In [6]:
# %pip install symspellpy

In [7]:
from symspellpy import SymSpell, Verbosity
import pkg_resources

sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)

dictionary_path = pkg_resources.resource_filename(
    "symspellpy",
    "frequency_dictionary_en_82_765.txt"
)

sym_spell.load_dictionary(dictionary_path, term_index=0, count_index=1)


CHAR_FIXES = {
    "0": "O",
    "1": "I",
    "3": "E",
    "4": "A",
    "5": "S",
    "8": "B",
}

def normalize_raw_word(raw_word):
    fixed = ""
    for char in raw_word.upper():
        fixed += CHAR_FIXES.get(char, char)
    return fixed

def correct_word(raw_word):
    normalized = normalize_raw_word(raw_word)
    suggestions = sym_spell.lookup(
        normalized.lower(),
        Verbosity.CLOSEST,
        max_edit_distance=2,
    )
    if suggestions:
        return suggestions[0].term.upper()
    return normalized

def get_correction_suggestions(raw_word, limit=3):
    normalized = normalize_raw_word(raw_word)
    suggestions = sym_spell.lookup(
        normalized.lower(),
        Verbosity.ALL,
        max_edit_distance=2,
    )
    words = []
    for suggestion in suggestions:
        word = suggestion.term.upper()
        if word not in words:
            words.append(word)
        if len(words) >= limit:
            break
    return words

C:\Users\HP\AppData\Local\Temp\ipykernel_6512\1318905178.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [8]:
# def main():
#     cap = None
#     try:
#         WIDTH, HEIGHT = 1280, 720
#         cap = cv2.VideoCapture(0)
#         cap.set(3, WIDTH)
#         cap.set(4, HEIGHT)
#         img_canvas = np.zeros((HEIGHT, WIDTH, 3), np.uint8)
        
#         # cap = cv2.VideoCapture(0)
#         detector = handDetector(num_hands=1)
    
#         xp, yp = 0, 0
#         brush_thickness = 6
#         draw_color = (255, 0, 255)
        
#         last_prediction = ""
#         last_raw_word = ""
#         # last_suggested_word = ""
#         last_suggested_words = []
    
#         last_draw_time = None
#         is_word_finalized = False
#         current_sentence = []
#         next_word_suggestions = []
#         pause_seconds = 3.0

#         window_name = "Adult Air Writing Test"
#         while True:
#             success, img = cap.read()
#             if not success:
#                 print("Camera not found")
#                 break

#             img = cv2.flip(img, 1)
#             img = cv2.resize(img, (WIDTH, HEIGHT))
#             height, width = HEIGHT, WIDTH
    
#             img = detector.findHands(img)
#             lm_list = detector.findPosition(img, draw=False)
    
#             if lm_list:
#                 x1, y1 = lm_list[8][1:]
#                 fingers = detector.fingersUp()
    
#                 # Index finger only: draw
#                 if fingers == [0, 1, 0, 0, 0]:
#                     cv2.circle(img, (x1, y1), 13, draw_color, cv2.FILLED)
#                     last_draw_time = time.time()
#                     is_word_finalized = False
#                     last_suggested_words = []
    
#                     if xp == 0 and yp == 0:
#                         xp, yp = x1, y1
    
#                     cv2.line(img, (xp, yp), (x1, y1), draw_color, brush_thickness)
#                     cv2.line(img_canvas, (xp, yp), (x1, y1), draw_color, brush_thickness)
#                     xp, yp = x1, y1
#                 else:
#                     xp, yp = 0, 0
    
#                 # Full open hand: erase
#                 if fingers == [1, 1, 1, 1, 1]:
#                     erase_x, erase_y = lm_list[9][1], lm_list[9][2]
#                     cv2.circle(img_canvas, (erase_x, erase_y), 100, (0, 0, 0), cv2.FILLED)
    
#             img_gray = cv2.cvtColor(img_canvas, cv2.COLOR_BGR2GRAY)
#             _, img_inv = cv2.threshold(img_gray, 50, 255, cv2.THRESH_BINARY_INV)
#             img_inv = cv2.cvtColor(img_inv, cv2.COLOR_GRAY2BGR)
    
#             img = cv2.bitwise_and(img, img_inv)
#             img = cv2.bitwise_or(img, img_canvas)
    
#             if (
#                 last_draw_time is not None
#                 and not is_word_finalized
#                 and canvas_has_content(img_canvas)
#                 and time.time() - last_draw_time >= pause_seconds
#             ):
#                 word, confidence, predictions = predict_word_canvas (
#                     img_canvas,
#                     alphabet_model,
#                     alphabet_classes,
#                 )
            
#                 if word:
#                     suggested_words = get_correction_suggestions(word)
#                     current_sentence.append(word)
                
#                     last_raw_word = word
#                     last_suggested_words = [
#                         suggestion for suggestion in suggested_words
#                         if suggestion.upper() != word.upper()
#                     ][:3]

#                     sentence_text = " ".join(current_sentence)
#                     next_word_suggestions = get_next_word_suggestions(sentence_text)
            
                
#                     if last_suggested_words:
#                         last_prediction = f"Word: {word} ({confidence * 100:.2f}%)"
#                     else:
#                         last_prediction = f"Word: {word} | No suggestions. Press 0 to rewrite"

#                     print(last_prediction)
#                     print("Spelling suggestions:", last_suggested_words)
#                     print("Next word suggestions:", next_word_suggestions)
#                     print("Sentence:", sentence_text)

#                 img_canvas[:] = 0
#                 xp, yp = 0, 0
#                 is_word_finalized = True

#             instruction_text = "Index: draw | pause 3s: word | C: clear canvas | R: reset sentence | Q: quit"
#             # suggestion_text = f"Suggestions: {suggestion_text} | 1/2/3 accept | 0 rewrite"
#             cv2.putText(
#                 img,
#                 instruction_text,
#                 (20, 650),
#                 cv2.FONT_HERSHEY_SIMPLEX,
#                 0.6,
#                 (0, 0, 255),
#                 2,
#             )
#             if last_prediction:
#                 cv2.putText(
#                     img,
#                     last_prediction,
#                     (20, 40),
#                     cv2.FONT_HERSHEY_SIMPLEX,
#                     0.55,
#                     (0, 0, 0),
#                     2,
#                 )
#             sentence_text = " ".join(current_sentence)
#             if sentence_text:
#                 cv2.putText(
#                     img,
#                     f"Sentence: {sentence_text}",
#                     (20, 75),
#                     cv2.FONT_HERSHEY_SIMPLEX,
#                     0.6,
#                     (0, 128, 0),
#                     2,
#                 )

#             if last_suggested_words:
#                 suggestion_items = " | ".join(
#                     f"{index + 1}:{suggestion}"
#                     for index, suggestion in enumerate(last_suggested_words)
#                 )
            
#                 suggestion_text = f"Suggestions: {suggestion_items} | 1/2/3 accept | 0 rewrite"
            
#                 font = cv2.FONT_HERSHEY_SIMPLEX
#                 font_scale = 0.55
#                 thickness = 2
#                 padding = 12
            
#                 text_size, _ = cv2.getTextSize(suggestion_text,font,font_scale,thickness,)
#                 text_width, text_height = text_size
            
#                 box_x2 = WIDTH - 20
#                 box_y1 = 20
#                 box_x1 = box_x2 - text_width - padding * 2
#                 box_y2 = box_y1 + text_height + padding * 2
            
#                 cv2.rectangle(img,(box_x1, box_y1),(box_x2, box_y2),(255, 255, 255),cv2.FILLED)
#                 cv2.rectangle(img,(box_x1, box_y1),(box_x2, box_y2),(255, 120, 0),2)

#                 cv2.putText(
#                     img,
#                     suggestion_text,
#                     (box_x1 + padding, box_y2 - padding),
#                     font,
#                     font_scale,
#                     (255, 120, 0),
#                     thickness,
#                 )

#             if next_word_suggestions:
#                 gen_text = " | ".join(next_word_suggestions)
#                 cv2.putText(
#                     img,
#                     f"GenAI next: {gen_text}",
#                     (20, 115),
#                     cv2.FONT_HERSHEY_SIMPLEX,
#                     0.6,
#                     (255, 0, 120),
#                     2,
#                 )
            
#             cv2.imshow(window_name, img)
#             # cv2.imshow("Canvas Only", img_canvas)

            
#             key = cv2.waitKey(1) & 0xFF
#             if key == ord("d"):
#                 label, confidence = predict_canvas(img_canvas, digit_model, digit_classes)
#                 last_prediction = f"Digit: {label} ({confidence * 100:.2f}%)"
#                 print(last_prediction)
        
#             if key == ord("a"):
#                 label, confidence = predict_canvas(img_canvas, alphabet_model, alphabet_classes)
#                 last_prediction = f"Alphabet: {label} ({confidence * 100:.2f}%)"
#                 print(last_prediction)
    
#             if key == ord("c"):
#                 img_canvas[:] = 0
#                 xp, yp = 0, 0
#                 last_prediction = ""
#                 last_draw_time = None
#                 is_word_finalized = True
    
#             if key == ord("r"):
#                 current_sentence = []
#                 next_word_suggestions = []
#                 last_prediction = ""
#                 last_raw_word = ""
#                 last_suggested_words = []
#                 print("Sentence reset")

#             if key in [ord("1"), ord("2"), ord("3")]:
#                 index = int(chr(key)) - 1
            
#                 if index < len(last_suggested_words) and current_sentence:
#                     chosen_word = last_suggested_words[index]
#                     current_sentence[-1] = chosen_word
#                     last_prediction = f"Replaced: {last_raw_word} -> {chosen_word}"
            
#                     last_raw_word = ""
#                     last_suggested_words = []

#                     print(last_prediction)
#                     print("Sentence:", " ".join(current_sentence))

#             if key == ord("0"):
#                 if current_sentence:
#                     removed_word = current_sentence.pop()
            
#                     img_canvas[:] = 0
#                     xp, yp = 0, 0
#                     last_prediction = f"Rewrite: removed {removed_word}"
#                     last_raw_word = ""
#                     last_suggested_words = []
#                     next_word_suggestions = []
#                     last_draw_time = None
#                     is_word_finalized = True
            
#                     print(last_prediction)
#                     print("Sentence:", " ".join(current_sentence))

            
#             if key == ord("q"):
#                 break
        
#     finally:
#         if cap is not None:
#             cap.release()
#         cv2.destroyAllWindows()


# if __name__ == "__main__":
#     main()

In [9]:
# %pip install transformers

In [10]:
from transformers import pipeline

next_word_generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

C:\Users\HP\Desktop\Placement\Air-Writing-and-Character-Recognition\newProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|████████████████████████████████████████████████████████████████████████████| 76/76 [00:00<00:00, 1287.37it/s]


In [12]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "AventIQ-AI/gpt2-next-word-prediction"
nw_model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
nw_tokenizer = GPT2Tokenizer.from_pretrained(model_name)

import os

def load_title_list(file_path="entertainment_training_data.txt"):
    if not os.path.exists(file_path):
        print(f"Warning: {file_path} not found, title suggestions disabled")
        return []
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line.strip().upper() for line in f if line.strip()]
    print(f"Loaded {len(lines)} entertainment titles")
    return lines

TITLE_LIST = load_title_list()

def search_titles_by_prefix(prefix, titles, limit=3):
    prefix = prefix.strip().upper()
    if not prefix:
        return []

    if len(entertainment_suggestions) < limit and len(last_word) > 2:
        counter = Counter()

         # first pass: lines that start exactly with this prefix
        for line in titles:
            if line.startswith(prefix) and line != prefix:
                remainder = line[len(prefix):].strip()
                if not remainder:
                    continue
                next_word = remainder.split()[0]
                next_word_clean = "".join(c for c in next_word if c.isalpha())
                if len(next_word_clean) >= 3:
                    counter[next_word_clean] += 2 

        # second pass: lines that contain the prefix anywhere
        for line in titles:
            if prefix in line and not line.startswith(prefix):
                idx = line.index(prefix)
                remainder = line[idx + len(prefix):].strip()
                if not remainder:
                    continue
                next_word = remainder.split()[0]
                next_word_clean = "".join(c for c in next_word if c.isalpha())
                if len(next_word_clean) >= 3:
                    counter[next_word_clean] += 1
    
        top = [word for word, count in counter.most_common(limit)]
        return top

def search_titles_by_prefix(prefix, titles, limit=3):
    prefix = prefix.strip().upper()
    if not prefix:
        return []

    counter = Counter()

    for line in titles:
        if line.startswith(prefix) and line != prefix:
            remainder = line[len(prefix):].strip()
            if not remainder:
                continue
            next_word = remainder.split()[0]
            next_word_clean = "".join(c for c in next_word if c.isalpha())
            if len(next_word_clean) >= 3:
                counter[next_word_clean] += 2

    top = [word for word, count in counter.most_common(limit)]
    return top


def get_next_word_suggestions(sentence, limit=3):
    if not sentence.strip():
        return []

    sentence_upper = sentence.strip().upper()
    last_word = sentence_upper.split()[-1]

    entertainment_suggestions = []
    seen = set()

    # full sentence startswith match
    for line in TITLE_LIST:
        if line.startswith(sentence_upper) and line != sentence_upper:
            remainder = line[len(sentence_upper):].strip()
            if not remainder:
                continue
            next_word = remainder.split()[0]
            next_word_clean = "".join(c for c in next_word if c.isalpha())
            if len(next_word_clean) >= 3 and next_word_clean not in seen:
                seen.add(next_word_clean)
                entertainment_suggestions.append(next_word_clean)
        if len(entertainment_suggestions) >= limit:
            break

    # last word fallback only if last word is longer than 2 characters
    if len(entertainment_suggestions) < limit and len(last_word) > 2:
        counter = Counter()
        for line in TITLE_LIST:
            words = line.split()
            for i, word in enumerate(words):
                if word == last_word and i + 1 < len(words):
                    next_word_clean = "".join(c for c in words[i+1] if c.isalpha())
                    if len(next_word_clean) >= 3:
                        counter[next_word_clean] += 1
        for word, _ in counter.most_common(limit):
            if word not in seen:
                seen.add(word)
                entertainment_suggestions.append(word)
            if len(entertainment_suggestions) >= limit:
                break

    # GPT2 fills remaining slots
    remaining = limit - len(entertainment_suggestions)
    if remaining > 0:
        input_ids = nw_tokenizer.encode(sentence.strip(), return_tensors="pt").to(device)
        seen_gpt2 = set(entertainment_suggestions)
        with torch.no_grad():
            outputs = nw_model(input_ids)
            logits = outputs.logits[:, -1, :]
            top_k = torch.topk(logits, k=300, dim=-1)
        for idx in top_k.indices[0].tolist():
            word = nw_tokenizer.decode([idx], skip_special_tokens=True).strip()
            word_clean = "".join(c for c in word if c.isalpha()).upper()
            if len(word_clean) < 2:
                continue
            if word_clean not in seen_gpt2:
                seen_gpt2.add(word_clean)
                entertainment_suggestions.append(word_clean)
            if len(entertainment_suggestions) >= limit:
                break

    return entertainment_suggestions[:limit]     

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████| 148/148 [00:00<00:00, 1699.54it/s]


Loaded 82770 entertainment titles


In [ ]:
print("Testing model:")
for test in ["if"]:
    sugs = get_next_word_suggestions(test, limit=5)
    print(f"{test!r} -> {sugs}")

# paste all your imports and model loading code above, then run this

# test_cases = [
#     "AVENGERS",
#     "HARRY",
#     "SHAH",
#     "I LOVE",
#     "DILWALE",
#     "STRANGER",
#     "MONEY",
#     "DARK",
#     "BREAKING",
#     "THE",
# ]

# print("Testing get_next_word_suggestions:\n")
# for sentence in test_cases:
#     suggestions = get_next_word_suggestions(sentence, limit=3)
#     print(f"{sentence!r} -> {suggestions}")

In [ ]:
def main():
    cap = None
    try:
        WIDTH, HEIGHT = 1280, 720
        cap = cv2.VideoCapture(0)
        cap.set(3, WIDTH)
        cap.set(4, HEIGHT)
        img_canvas = np.zeros((HEIGHT, WIDTH, 3), np.uint8)
        
        # cap = cv2.VideoCapture(0)
        detector = handDetector(num_hands=1)
    
        xp, yp = 0, 0
        brush_thickness = 6
        draw_color = (255, 0, 255)
        
        last_prediction = ""
        last_raw_word = ""
        # last_suggested_word = ""
        last_suggested_words = []
    
        last_draw_time = None
        is_word_finalized = False
        current_sentence = []
        next_word_suggestions = []
        pause_seconds = 3.0

        window_name = "Adult Air Writing Test"
        while True:
            success, img = cap.read()
            if not success:
                print("Camera not found")
                break

            img = cv2.flip(img, 1)
            img = cv2.resize(img, (WIDTH, HEIGHT))
            height, width = HEIGHT, WIDTH
    
            img = detector.findHands(img)
            lm_list = detector.findPosition(img, draw=False)
    
            if lm_list:
                x1, y1 = lm_list[8][1:]
                fingers = detector.fingersUp()
    
                # Index finger only: draw
                if fingers == [0, 1, 0, 0, 0]:
                    cv2.circle(img, (x1, y1), 13, draw_color, cv2.FILLED)
                    last_draw_time = time.time()
                    is_word_finalized = False
                    last_suggested_words = []
    
                    if xp == 0 and yp == 0:
                        xp, yp = x1, y1
    
                    cv2.line(img, (xp, yp), (x1, y1), draw_color, brush_thickness)
                    cv2.line(img_canvas, (xp, yp), (x1, y1), draw_color, brush_thickness)
                    xp, yp = x1, y1
                else:
                    xp, yp = 0, 0
    
                # Full open hand: erase
                if fingers == [1, 1, 1, 1, 1]:
                    erase_x, erase_y = lm_list[9][1], lm_list[9][2]
                    cv2.circle(img_canvas, (erase_x, erase_y), 100, (0, 0, 0), cv2.FILLED)
    
            img_gray = cv2.cvtColor(img_canvas, cv2.COLOR_BGR2GRAY)
            _, img_inv = cv2.threshold(img_gray, 50, 255, cv2.THRESH_BINARY_INV)
            img_inv = cv2.cvtColor(img_inv, cv2.COLOR_GRAY2BGR)
    
            img = cv2.bitwise_and(img, img_inv)
            img = cv2.bitwise_or(img, img_canvas)
    
            if (
                last_draw_time is not None
                and not is_word_finalized
                and canvas_has_content(img_canvas)
                and time.time() - last_draw_time >= pause_seconds
            ):
                word, confidence, predictions = predict_word_canvas (
                    img_canvas,
                    alphabet_model,
                    alphabet_classes,
                )
            
                if word:
                    suggested_words = get_correction_suggestions(word)
                    current_sentence.append(word)
                
                    last_raw_word = word
                    last_suggested_words = [
                        suggestion for suggestion in suggested_words
                        if suggestion.upper() != word.upper()
                    ][:3]

                    sentence_text = " ".join(current_sentence)
                    next_word_suggestions = get_next_word_suggestions(sentence_text)
            
                
                    if last_suggested_words:
                        last_prediction = f"Word: {word} ({confidence * 100:.2f}%)"
                    else:
                        last_prediction = f"Word: {word} | No suggestions. Press 0 to rewrite"

                    print(last_prediction)
                    print("Spelling suggestions:", last_suggested_words)
                    print("Next word suggestions:", next_word_suggestions)
                    print("Sentence:", sentence_text)

                img_canvas[:] = 0
                xp, yp = 0, 0
                is_word_finalized = True

            instruction_text = "Index: draw | pause 3s: word | C: clear canvas | R: reset sentence | Q: quit"
            # suggestion_text = f"Suggestions: {suggestion_text} | 1/2/3 accept | 0 rewrite"
            cv2.putText(
                img,
                instruction_text,
                (20, 650),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2,
            )
            if last_prediction:
                cv2.putText(
                    img,
                    last_prediction,
                    (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.55,
                    (0, 0, 0),
                    2,
                )
            sentence_text = " ".join(current_sentence)
            if sentence_text:
                cv2.putText(
                    img,
                    f"Sentence: {sentence_text}",
                    (20, 75),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 128, 0),
                    2,
                )

            if last_suggested_words:
                suggestion_items = " | ".join(
                    f"{index + 1}:{suggestion}"
                    for index, suggestion in enumerate(last_suggested_words)
                )
            
                suggestion_text = f"Suggestions: {suggestion_items} | 1/2/3 accept | 0 rewrite"
            
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 0.55
                thickness = 2
                padding = 12
            
                text_size, _ = cv2.getTextSize(suggestion_text,font,font_scale,thickness,)
                text_width, text_height = text_size
            
                box_x2 = WIDTH - 20
                box_y1 = 20
                box_x1 = box_x2 - text_width - padding * 2
                box_y2 = box_y1 + text_height + padding * 2
            
                cv2.rectangle(img,(box_x1, box_y1),(box_x2, box_y2),(255, 255, 255),cv2.FILLED)
                cv2.rectangle(img,(box_x1, box_y1),(box_x2, box_y2),(255, 120, 0),2)

                cv2.putText(
                    img,
                    suggestion_text,
                    (box_x1 + padding, box_y2 - padding),
                    font,
                    font_scale,
                    (255, 120, 0),
                    thickness,
                )

            if next_word_suggestions:
                items = " | ".join(
                    f"{index + 4}:{word}"
                    for index, word in enumerate(next_word_suggestions)
                )
                cv2.putText(
                    img,
                    f"Next: {items}",
                    (20, 115),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255, 0, 120),
                    2,
                )
            
            cv2.imshow(window_name, img)
            # cv2.imshow("Canvas Only", img_canvas)

            
            key = cv2.waitKey(1) & 0xFF
            if key == ord("d"):
                label, confidence = predict_canvas(img_canvas, digit_model, digit_classes)
                last_prediction = f"Digit: {label} ({confidence * 100:.2f}%)"
                print(last_prediction)
        
            if key == ord("a"):
                label, confidence = predict_canvas(img_canvas, alphabet_model, alphabet_classes)
                last_prediction = f"Alphabet: {label} ({confidence * 100:.2f}%)"
                print(last_prediction)
    
            if key == ord("c"):
                img_canvas[:] = 0
                xp, yp = 0, 0
                last_prediction = ""
                last_draw_time = None
                is_word_finalized = True
    
            if key == ord("r"):
                current_sentence = []
                next_word_suggestions = []
                last_prediction = ""
                last_raw_word = ""
                last_suggested_words = []
                print("Sentence reset")

            if key in [ord("1"), ord("2"), ord("3")]:
                index = int(chr(key)) - 1
                if index < len(last_suggested_words) and current_sentence:
                    chosen_word = last_suggested_words[index]
                    current_sentence[-1] = chosen_word
                    sentence_text = " ".join(current_sentence)
                    next_word_suggestions = get_next_word_suggestions(sentence_text)
                    last_prediction = f"Replaced: {last_raw_word} -> {chosen_word}"
                    last_raw_word = ""
                    last_suggested_words = []
                    print(last_prediction)
                    print("Sentence:", " ".join(current_sentence))

            if key in [ord("4"), ord("5"), ord("6"), ord("7")]:
                index = int(chr(key)) - 4

                print(f"Key pressed: {chr(key)}")
                print(f"next_word_suggestions: {next_word_suggestions}")
                index = int(chr(key)) - 4
                print(f"index: {index}")
            
                if index < len(next_word_suggestions):
                    chosen_word = next_word_suggestions[index]
                    current_sentence.append(chosen_word)
                    sentence_text = " ".join(current_sentence)
                    next_word_suggestions = get_next_word_suggestions(sentence_text)
            
                    last_prediction = f"Added: {chosen_word}"
                    last_raw_word = chosen_word
                    last_suggested_words = []
            
                    print(last_prediction)
                    print("Sentence:", " ".join(current_sentence))


            if key == ord("0"):
                if current_sentence:
                    removed_word = current_sentence.pop()
            
                    img_canvas[:] = 0
                    xp, yp = 0, 0
                    last_prediction = f"Rewrite: removed {removed_word}"
                    last_raw_word = ""
                    last_suggested_words = []
                    next_word_suggestions = []
                    last_draw_time = None
                    is_word_finalized = True
            
                    print(last_prediction)
                    print("Sentence:", " ".join(current_sentence))

            
            if key == ord("q"):
                break
        
    finally:
        if cap is not None:
            cap.release()
        cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "./models/gpt2-entertainment"
nw_model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
nw_tokenizer = GPT2Tokenizer.from_pretrained(model_name)

print("Tokenizer vocab size:", nw_tokenizer.vocab_size)
print("Model config:", nw_model.config)

In [ ]:
import pandas as pd
df = pd.read_csv("bollywood_movies.csv")
print(df.columns.tolist())
print(df.head(2))

In [ ]:
import pandas as pd
df = pd.read_csv("Hindi TV Serials.csv")
print(df.columns.tolist())
print(df.head(2))

In [ ]:
import pandas as pd
df = pd.read_csv("IMDb Movies India.csv", encoding="latin1")
print(df.columns.tolist())
print(df.head(2))

In [ ]:
import pandas as pd
df = pd.read_csv("indian_movies.csv")
print(df.columns.tolist())
print(df.head(2))

In [ ]:
import pandas as pd
df = pd.read_csv("indian_tv_series.csv")
print(df.columns.tolist())
print(df.head(2))

In [ ]:
import pandas as pd
df = pd.read_csv("movies.csv")
print(df.columns.tolist())
print(df.head(2))

In [ ]:
import pandas as pd
df = pd.read_csv("netflix_titles.csv")
print(df.columns.tolist())
print(df.head(2))

In [ ]:
# import pandas as pd
# import re

# df_bollywood = pd.read_csv("bollywood_movies.csv")
# df_hindi_tv = pd.read_csv("Hindi TV Serials.csv")
# df_indian_movies = pd.read_csv("indian_movies.csv")
# df_indian_tv = pd.read_csv("indian_tv_series.csv")
# df_movies = pd.read_csv("movies.csv")
# df_netflix = pd.read_csv("netflix_titles.csv")
# df_imdb_india = pd.read_csv("IMDb Movies India.csv", encoding="latin1")


In [ ]:
# lines = []

# for _, row in df_bollywood.dropna(subset=["title"]).iterrows():
#     lines.append(str(row["title"]).strip())

# for _, row in df_hindi_tv.dropna(subset=["Name"]).iterrows():
#     name = str(row["Name"]).strip()
#     cast = str(row["Cast"]).strip() if pd.notna(row["Cast"]) else ""
#     lines.append(name)
#     if cast:
#         lines.append(f"{name} {cast}")

# for _, row in df_indian_movies.dropna(subset=["title"]).iterrows():
#     title = str(row["title"]).strip()
#     cast = str(row["cast"]).strip() if pd.notna(row["cast"]) else ""
#     director = str(row["directors"]).strip() if pd.notna(row["directors"]) else ""
#     lines.append(title)
#     if cast:
#         lines.append(f"{title} {cast}")
#     if director:
#         lines.append(f"{title} {director}")

# for _, row in df_indian_tv.dropna(subset=["title"]).iterrows():
#     title = str(row["title"]).strip()
#     cast = str(row["cast"]).strip() if pd.notna(row["cast"]) else ""
#     lines.append(title)
#     if cast:
#         lines.append(f"{title} {cast}")

# for _, row in df_movies.dropna(subset=["Film Name"]).iterrows():
#     name = str(row["Film Name"]).strip()
#     # remove leading numbering like "1. Animal" -> "Animal"
#     if ". " in name:
#         name = name.split(". ", 1)[1]
#     lines.append(name)

# for _, row in df_netflix.dropna(subset=["title"]).iterrows():
#     title = str(row["title"]).strip()
#     cast = str(row["cast"]).strip() if pd.notna(row["cast"]) else ""
#     director = str(row["director"]).strip() if pd.notna(row["director"]) else ""
#     lines.append(title)
#     if cast:
#         lines.append(f"{title} {cast}")
#     if director:
#         lines.append(f"{title} {director}")

# for _, row in df_imdb_india.dropna(subset=["Name"]).iterrows():
#     name = str(row["Name"]).strip()
#     director = str(row["Director"]).strip() if pd.notna(row["Director"]) else ""
#     a1 = str(row["Actor 1"]).strip() if pd.notna(row["Actor 1"]) else ""
#     a2 = str(row["Actor 2"]).strip() if pd.notna(row["Actor 2"]) else ""
#     a3 = str(row["Actor 3"]).strip() if pd.notna(row["Actor 3"]) else ""
#     actors = " ".join(filter(None, [a1, a2, a3]))
#     lines.append(name)
#     if actors:
#         lines.append(f"{name} {actors}")
#     if director:
#         lines.append(f"{name} {director}")

# def clean_line(text):
#     # remove anything inside brackets like (2019), [2020]
#     text = re.sub(r"[\(\[\{].*?[\)\]\}]", "", text)
#     # remove special characters except letters, numbers, spaces, commas
#     text = re.sub(r"[^a-zA-Z0-9\s,]", "", text)
#     # collapse multiple spaces into one
#     text = re.sub(r"\s+", " ", text)
#     return text.strip()

# cleaned = []
# for line in lines:
#     line = clean_line(line)
#     # skip lines shorter than 3 characters
#     if len(line) < 3:
#         continue
#     cleaned.append(line)


# seen = set()
# unique_lines = []
# for line in cleaned:
#     lower = line.lower()
#     if lower not in seen:
#         seen.add(lower)
#         unique_lines.append(line)

# print(f"Total lines before dedup: {len(cleaned)}")
# print(f"Total lines after dedup: {len(unique_lines)}")
# print("Sample lines:")
# for line in unique_lines[:10]:
#     print(line)

# with open("entertainment_training_data.txt", "w", encoding="utf-8") as f:
#     for line in unique_lines:
#         f.write(line + "\n")

# print("Done! Saved to entertainment_training_data.txt")

In [ ]:
# %pip install transformers datasets accelerate

In [ ]:
# import torch
# from transformers import GPT2LMHeadModel, GPT2Tokenizer, GPT2Config
# from transformers import TextDataset, DataCollatorForLanguageModeling
# from transformers import Trainer, TrainingArguments

# model_name = "AventIQ-AI/gpt2-next-word-prediction"
# tokenizer = GPT2Tokenizer.from_pretrained(model_name)
# model = GPT2LMHeadModel.from_pretrained(model_name)

# tokenizer.pad_token = tokenizer.eos_token

# # ---- Load dataset ----
# def load_dataset(file_path, tokenizer, block_size=64):
#     dataset = TextDataset(
#         tokenizer=tokenizer,
#         file_path=file_path,
#         block_size=block_size
#     )
#     return dataset

# train_dataset = load_dataset("entertainment_training_data.txt", tokenizer)

# data_collator = DataCollatorForLanguageModeling(
#     tokenizer=tokenizer,
#     mlm=False
# )

# # ---- Training arguments ----
# training_args = TrainingArguments(
#     output_dir="./gpt2-entertainment",
#     overwrite_output_dir=True,
#     num_train_epochs=5,
#     per_device_train_batch_size=8,
#     save_steps=500,
#     save_total_limit=2,
#     logging_steps=100,
#     prediction_loss_only=True,
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     data_collator=data_collator,
#     train_dataset=train_dataset,
# )

# # ---- Train ----
# trainer.train()

# # ---- Save fine-tuned model ----
# model.save_pretrained("./gpt2-entertainment")
# tokenizer.save_pretrained("./gpt2-entertainment")

# print("Fine-tuning complete! Model saved to ./gpt2-entertainment")